# Reading the Acre Road solar flux files

How to read the Acre Road SRT's daily solar flux files with `read_solar_flux.py`,
which sits beside this notebook, and what is in them. Run it from the
`examples/` folder of the products, or from `receiver_scheduler/solar_flux_examples/`
on the observatory computer. It needs numpy, matplotlib and astropy, and no
network.

1. One day's FITS file: header, runs, flux against time.
2. The quality flags, and what the reduction took out (scallop, atmosphere).
3. The brightest half hour of the day, at full resolution.
4. The same day's RSTN3 file, which is the same data rounded to whole SFU.
5. Every day in the folder against the RSTN stations' noon values.

In [ ]:
import sys
from pathlib import Path
from datetime import timezone

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import read_solar_flux as rsf      # the reader beside this notebook

# The products folder: one up from examples/, or data/solar_flux/ from the
# copy kept in git.
HERE = Path.cwd()
for folder in (HERE.parent, HERE.parent / "data" / "solar_flux", HERE):
    if (folder / "fits").is_dir():
        break
else:
    raise FileNotFoundError("no fits/ folder found next to this notebook")
FITS = sorted((folder / "fits").glob("acre_srt_1417MHz_*.fits"))
print("%d days in %s, %s to %s" % (len(FITS), folder.resolve(), FITS[0].stem[-8:], FITS[-1].stem[-8:]))

# Series colours, fixed per entity whatever is plotted.
OURS = "#2a78d6"
STATION = {"Learmonth": "#eb6834", "Palehua": "#1baf7a", "San Vito": "#eda100"}
FLAGGED = "#9a9a96"
UTC = mdates.DateFormatter("%H:%M", tz=timezone.utc)

## 1. One day

`read_fits` returns the record times as UTC datetimes, the `FLUX` table (one
row per 3 s record), the `RUNS` table (one row per recording), and the primary
header. Pick a day by changing `DAY`. The default is the latest file.

In [ ]:
DAY = FITS[-1].stem[-8:]          # e.g. "20261007"
path = folder / "fits" / ("acre_srt_1417MHz_%s.fits" % DAY)
times, flux, runs, header = rsf.read_fits(path)

print(header["TELESCOP"], "|", header["DATE-OBS"], "to", header["DATE-END"])
print("band %.2f MHz, %.1f MHz wide" % (header["FREQ"] / 1e6, header["BANDWID"] / 1e6))
print("RSTN 1415 MHz noon:", {header.comments[k].split(", ")[-1]: header[k] for k in header if k.startswith("R_")})
runs["FILE", "OBS_NAME", "NROWS", "MEANFLUX", "SIGMA", "TSYS", "BEAM_SA", "SCALSOL", "MOD_PRE", "MOD_POST"]

In [ ]:
good = flux["QUALITY"] == 0

fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(times[~good], flux["FLUX"][~good], ".", ms=2, color=FLAGGED, label="flagged")
ax.plot(times[good], flux["FLUX"][good], ".", ms=2, color=OURS, label="Acre Road, %.2f MHz" % (header["FREQ"] / 1e6))
for key in header:
    if key.startswith("R_"):
        name = header.comments[key].split(", ")[-1]
        ax.axhline(header[key], lw=1.5, ls="--", color=STATION.get(name, FLAGGED), label="%s noon, 1415 MHz" % name)
ax.set_ylim(np.nanpercentile(flux["FLUX"][good], 0.5) - 10, np.nanpercentile(flux["FLUX"][good], 99.9) + 5)
ax.xaxis.set_major_formatter(UTC)
ax.set_xlabel("UTC on %s-%s-%s" % (DAY[:4], DAY[4:6], DAY[6:]))
ax.set_ylabel("Solar flux (SFU)")
ax.grid(alpha=0.3)
ax.legend(fontsize=8, loc="upper left")
ax.set_title("Solar flux above the atmosphere, tracking scallop removed")
plt.show()

## 2. Flags, and what the reduction took out

`QUALITY` is a bit mask: **1** for the first 5 min after the receiver starts
(it reads 0.3-1.7% high while it settles), **2** for the beam off the Sun (a
slew, or the stow at the end of a run), and **4** for samples dropped by the
radio. Flagged rows stay in the file. Radio bursts are never flagged.

`SCALLOP` is the gain the 0.5° drive steps cost and that was divided out of the
source term. `TRANSMIS` is the atmosphere's transmission, divided out to give
the flux above it. `TANT` is the antenna temperature before either.

In [ ]:
q = flux["QUALITY"]
for bit, what in ((1, "receiver settling"), (2, "beam off the Sun"), (4, "samples dropped")):
    print("bit %d  %-18s %5d rows" % (bit, what, int(((q & bit) > 0).sum())))
print("unflagged            %5d of %d" % (int(good.sum()), len(q)))

fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
a1.plot(times[good], 100 * (1 - flux["SCALLOP"][good]), ".", ms=1.5, color=OURS)
a1.set_ylabel("scallop loss (%)")
a1.set_title("Corrections applied per record")
a2.plot(times[good], 100 * (1 - flux["TRANSMIS"][good]), ".", ms=1.5, color=OURS)
a2.set_ylabel("atmospheric loss (%)")
for a in (a1, a2):
    a.grid(alpha=0.3)
a2.xaxis.set_major_formatter(UTC)
a2.set_xlabel("UTC")
plt.show()

## 3. The brightest half hour

The 30 minutes with the highest median flux, at full 3 s resolution, with
the run's statistical error as a band. `ERROR` is the run's record-to-record
scatter: radiometer noise plus the receiver's gain wobble, and nothing
systematic.

In [ ]:
t_s = flux["TIME"]
window = 1800.0
# Windows stepped by a minute, kept only where they are at least 80% full, so
# the gap between two runs never wins.
need = 0.8 * window / np.nanmedian(flux["TIMEDEL"])
best, s0 = -np.inf, t_s[good].min()
for s in np.arange(t_s[good].min(), t_s[good].max() - window, 60.0):
    inside = good & (t_s >= s) & (t_s < s + window)
    if inside.sum() >= need and np.median(flux["FLUX"][inside]) > best:
        best, s0 = np.median(flux["FLUX"][inside]), s
sel = good & (t_s >= s0) & (t_s < s0 + window)

fig, ax = plt.subplots(figsize=(11, 4))
f, e = flux["FLUX"][sel], flux["ERROR"][sel]
ax.fill_between(times[sel], f - e, f + e, color=OURS, alpha=0.25, lw=0, label="± ERROR")
ax.plot(times[sel], f, lw=1.2, color=OURS, label="Acre Road")
ax.xaxis.set_major_formatter(UTC)
ax.set_xlabel("UTC")
ax.set_ylabel("Solar flux (SFU)")
ax.grid(alpha=0.3)
ax.legend(fontsize=8)
ax.set_title("Brightest 30 min of the day")
plt.show()

## 4. The RSTN3 file

The RSTN3 file is the same data in the archive format of the RSTN network:
one line per record, the 1415 MHz field in whole SFU, and blank for a flagged
record. `read_rstn` returns the times and the flux, with NaN where the field
is blank. The rounding is the price of the format: compare it with the run's
`SIGMA`.

In [ ]:
months = ("JAN", "FEB", "MAR", "APR", "MAY", "JUN", "JUL", "AUG", "SEP", "OCT", "NOV", "DEC")
rstn_path = folder / "rstn" / ("%s%s%s.ACR" % (DAY[6:], months[int(DAY[4:6]) - 1], DAY[2:4]))
r_times, r_sfu = rsf.read_rstn(rstn_path)
print(rstn_path.name, "-", len(r_times), "lines,", int(np.isfinite(r_sfu).sum()), "with a value")
# One line with a value, between the quotes so the blank fields show.
print(repr(next(line for line in open(rstn_path, newline="") if line[36:42].strip())))

# Match each RSTN line to its FITS record by the UT second, then compare.
fits_sec = {int(round(t.timestamp())): v for t, v, ok in zip(times, flux["FLUX"], good) if ok}
diff = np.array([v - fits_sec[int(t.timestamp())] for t, v in zip(r_times, r_sfu)
                 if np.isfinite(v) and int(t.timestamp()) in fits_sec])
print("RSTN3 minus FITS: mean %+.3f SFU, rms %.3f SFU (rounding alone gives 0.289)" % (diff.mean(), diff.std()))
print("run scatter SIGMA:", ", ".join("%.2f" % s for s in runs["SIGMA"]), "SFU")

## 5. Every day in the folder

The daily median of the unflagged records, against the RSTN stations'
1415 MHz local-noon values from the same headers. The stations themselves
disagree by 10-20 SFU on a given day, which sets how closely any of them can be
matched. The feed was replaced on 2026-09-22, and each day is reduced with its
own feed's gain and beam (`TSYS`, `BEAM_SA` in RUNS).

In [ ]:
from datetime import datetime, timedelta
by_day = {datetime.strptime(p.stem[-8:], "%Y%m%d"): p for p in FITS}
# Every calendar day from the first to the last, NaN where there is no file,
# so a line never bridges days that were not observed.
days = [min(by_day) + timedelta(days=i) for i in range((max(by_day) - min(by_day)).days + 1)]
ours, ref = [], {name: [] for name in STATION}
for d in days:
    noon = {}
    value = np.nan
    if d in by_day:
        t, fl, rn, h = rsf.read_fits(by_day[d])
        ok = fl["QUALITY"] == 0
        if ok.any():
            value = np.nanmedian(fl["FLUX"][ok])
        noon = {h.comments[k].split(", ")[-1]: h[k] for k in h if k.startswith("R_")}
    ours.append(value)
    for name in STATION:
        ref[name].append(noon.get(name, np.nan))

fig, ax = plt.subplots(figsize=(11, 4.5))
for name, values in ref.items():
    ax.plot(days, values, "o-", ms=4, lw=1.5, color=STATION[name], label=name + " noon")
ax.plot(days, ours, "o-", ms=6, lw=2, color=OURS, label="Acre Road daily median")
ax.axvline(datetime(2026, 9, 22), color=FLAGGED, lw=1, ls=":")
ax.text(datetime(2026, 9, 22), 0.97, " feed replaced", fontsize=8, va="top", color="#55554f",
        transform=ax.get_xaxis_transform())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax.set_ylabel("Solar flux (SFU)")
ax.grid(alpha=0.3)
ax.legend(fontsize=8, ncol=4, loc="lower left")
ax.set_title("Daily solar flux, Acre Road 1417 MHz and RSTN 1415 MHz")
plt.show()